In [ ]:
# ========================================
# CELL 1: VERIFIKASI STRUKTUR DATASET
# ========================================
"""
JALANKAN CELL INI PERTAMA!
Verifikasi struktur folder dataset Kaggle sebelum training.
Dataset harus sudah ditambahkan via: Add Data → Your Datasets → crisismmd
"""

import os
from pathlib import Path

# ── Kaggle dataset root ──────────────────────────────────────────────────────
# Kaggle otomatis mount dataset di /kaggle/input/{dataset-slug}/
# Ganti 'crisismmd' jika slug dataset Anda berbeda (cek di URL dataset Anda)
KAGGLE_INPUT = '/kaggle/input/datasets/alieffathurrahman/crisismmd'

print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

# 1. Cek root dataset ada
if not os.path.exists(KAGGLE_INPUT):
    raise FileNotFoundError(
        f"\n❌ Dataset tidak ditemukan di: {KAGGLE_INPUT}\n"
        "Pastikan:\n"
        "  1. Dataset sudah diupload ke Kaggle (My Datasets → crisismmd)\n"
        "  2. Dataset sudah ditambahkan ke notebook: Add Data → Your Datasets\n"
        "  3. Slug dataset sesuai — cek URL: kaggle.com/datasets/<username>/crisismmd\n"
        "     Jika slug berbeda, ubah variable KAGGLE_INPUT di atas."
    )

print(f"✅ Root dataset ditemukan: {KAGGLE_INPUT}")

# 2. List semua isi root
print(f"\n📁 Isi root ({KAGGLE_INPUT}):")
for item in sorted(os.listdir(KAGGLE_INPUT)):
    full = os.path.join(KAGGLE_INPUT, item)
    kind = "📂" if os.path.isdir(full) else "📄"
    print(f"   {kind} {item}")

# 3. Cari folder CrisisMMD_v2.0 (bisa langsung di root atau di subfolder)
def find_dir(root, target):
    """Cari folder target secara rekursif."""
    for dirpath, dirnames, _ in os.walk(root):
        if target in dirnames:
            return os.path.join(dirpath, target)
    return None

CRISISMMMD_ROOT = None
# Coba langsung
if os.path.isdir(os.path.join(KAGGLE_INPUT, 'CrisisMMD_v2.0')):
    CRISISMMMD_ROOT = os.path.join(KAGGLE_INPUT, 'CrisisMMD_v2.0')
else:
    CRISISMMMD_ROOT = find_dir(KAGGLE_INPUT, 'CrisisMMD_v2.0')

if CRISISMMMD_ROOT is None:
    # Mungkin tidak ada subfolder, dataset langsung di root
    # Cek apakah folder data_image, annotations, crisismmd_datasplit_all ada di root
    candidate_dirs = ['data_image', 'annotations', 'crisismmd_datasplit_all']
    if all(os.path.isdir(os.path.join(KAGGLE_INPUT, d)) for d in candidate_dirs):
        CRISISMMMD_ROOT = KAGGLE_INPUT
        print(f"\n✅ Struktur dataset langsung di root (tanpa subfolder CrisisMMD_v2.0)")
    else:
        print(f"\n⚠️  Folder 'CrisisMMD_v2.0' tidak ditemukan.")
        print("    Folder yang tersedia di root:")
        for d in os.listdir(KAGGLE_INPUT):
            print(f"      - {d}")
        raise FileNotFoundError(
            "❌ Tidak dapat menemukan struktur CrisisMMD.\n"
            "   Pastikan zip dataset diekstrak dengan benar saat upload ke Kaggle."
        )

print(f"\n✅ CrisisMMD root: {CRISISMMMD_ROOT}")

# 4. Verifikasi sub-folder kunci
print("\n📁 Verifikasi sub-folder:")

DATA_IMAGE_PATH   = os.path.join(CRISISMMMD_ROOT, 'data_image')
ANNOTATIONS_PATH  = os.path.join(CRISISMMMD_ROOT, 'annotations')

# Datasplit bisa berada di nested folder
DATASPLIT_CANDIDATES = [
    os.path.join(CRISISMMMD_ROOT, 'crisismmd_datasplit_all', 'crisismmd_datasplit_all'),
    os.path.join(CRISISMMMD_ROOT, 'crisismmd_datasplit_all'),
]
DATASPLIT_PATH = None
for c in DATASPLIT_CANDIDATES:
    if os.path.isdir(c):
        tsv_files = [f for f in os.listdir(c) if f.endswith('.tsv')]
        if tsv_files:
            DATASPLIT_PATH = c
            break

for label, path in [
    ('data_image',   DATA_IMAGE_PATH),
    ('annotations',  ANNOTATIONS_PATH),
    ('datasplit',    DATASPLIT_PATH),
]:
    exists = path is not None and os.path.isdir(path)
    icon = "✅" if exists else "❌"
    print(f"   {icon} {label}: {path}")

if not all([
    os.path.isdir(DATA_IMAGE_PATH),
    os.path.isdir(ANNOTATIONS_PATH),
    DATASPLIT_PATH is not None,
]):
    raise FileNotFoundError("❌ Satu atau lebih sub-folder kritis tidak ditemukan. Cek struktur zip dataset.")

# 5. Preview isi tiap folder
print(f"\n📁 data_image — subfolder (maks 5):")
print(f"   {sorted(os.listdir(DATA_IMAGE_PATH))[:5]} ...")

print(f"\n📄 annotations — file TSV (maks 5):")
ann_files = [f for f in sorted(os.listdir(ANNOTATIONS_PATH)) if f.endswith('.tsv')]
print(f"   {ann_files[:5]} ...")
print(f"   Total TSV: {len(ann_files)}")

print(f"\n📄 datasplit — file TSV:")
split_files = sorted([f for f in os.listdir(DATASPLIT_PATH) if f.endswith('.tsv')])
for f in split_files:
    print(f"   - {f}")

# 6. Cek file datasplit wajib ada
REQUIRED_SPLITS = [
    'task_informative_text_img_train.tsv',
    'task_informative_text_img_dev.tsv',
    'task_informative_text_img_test.tsv',
    'task_humanitarian_text_img_train.tsv',
    'task_humanitarian_text_img_dev.tsv',
    'task_humanitarian_text_img_test.tsv',
]
print("\n📄 Verifikasi file datasplit wajib:")
missing = []
for req in REQUIRED_SPLITS:
    exists = req in split_files
    icon = "✅" if exists else "❌"
    print(f"   {icon} {req}")
    if not exists:
        missing.append(req)

if missing:
    print(f"\n⚠️  {len(missing)} file datasplit tidak ditemukan.")
    print("   Pastikan dataset lengkap sebelum lanjut training.")
else:
    print("\n✅ Semua file datasplit tersedia!")

# 7. Quick sanity check: baca satu file annotations
import pandas as pd
sample_ann = ann_files[0]
try:
    df_sample = pd.read_csv(os.path.join(ANNOTATIONS_PATH, sample_ann), sep='\t', encoding='latin-1')
    print(f"\n📊 Preview annotations ({sample_ann}):")
    print(f"   Shape: {df_sample.shape}")
    print(f"   Columns: {df_sample.columns.tolist()}")
    print(f"   Sample rows:")
    print(df_sample.head(2).to_string(index=False))
except Exception as e:
    print(f"\n⚠️  Gagal membaca file annotations: {e}")

print("\n" + "=" * 60)
print("✅ VERIFIKASI SELESAI — Aman lanjut ke Cell berikutnya!")
print("=" * 60)
print(f"\nPath yang akan digunakan:")
print(f"  BASE_PATH       = {CRISISMMMD_ROOT}")
print(f"  DATA_IMAGE_PATH = {DATA_IMAGE_PATH}")
print(f"  ANNOTATIONS_PATH= {ANNOTATIONS_PATH}")
print(f"  DATASPLIT_PATH  = {DATASPLIT_PATH}")

In [ ]:
# ========================================
# CELL 2: Install Libraries
# ========================================
"""
Install library tambahan yang belum ada di Kaggle
"""

import subprocess
subprocess.run(['pip', 'install', '-q', 'timm', 'torchmetrics', 'thop'], check=True)


In [ ]:
# ========================================
# CELL 3: Import Libraries
# ========================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import timm
import pandas as pd
import numpy as np
from PIL import Image
from typing import Dict, List, Tuple, Optional
import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
import json
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Output dir (Kaggle working directory)
OUTPUT_DIR = '/kaggle/working'
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
RESULTS_DIR    = os.path.join(OUTPUT_DIR, 'results')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"\nOutput directory: {OUTPUT_DIR}")

In [ ]:
# ========================================
# CELL 4: Configuration
# ========================================
"""
⚠️ CUSTOMIZATION — UBAH DI SINI ⚠️

SUBSET_SIZE:
  None  → Semua data (untuk final experiment)
  500   → 500 sample per split (testing cepat)

TASK:
  'informative'  → Binary (Informative vs Not Informative)
  'humanitarian' → Multi-class (7 kategori)
"""

SUBSET_SIZE = None        # ⚠️ Ubah: None untuk semua data
TASK        = 'informative'  # ⚠️ Ubah: 'informative' atau 'humanitarian'

# BASE_PATH ditetapkan dari Cell 1
BASE_PATH = CRISISMMMD_ROOT  # ← Otomatis dari verifikasi

TASK_CONFIG = {
    'informative': {
        'num_classes': 2,
        'label_col': 'image_info',
        'class_names': ['not_informative', 'informative'],
        'label_map': {'not_informative': 0, 'informative': 1}
    },
    'humanitarian': {
        'num_classes': 7,
        'label_col': 'image_human',
        'class_names': [
            'infrastructure_and_utility_damage', 'affected_individuals',
            'injured_or_dead_people', 'missing_or_found_people',
            'rescue_volunteering_or_donation_effort', 'vehicle_damage',
            'other_relevant_information'
        ],
        'label_map': {
            'infrastructure_and_utility_damage': 0, 'affected_individuals': 1,
            'injured_or_dead_people': 2, 'missing_or_found_people': 3,
            'rescue_volunteering_or_donation_effort': 4, 'vehicle_damage': 5,
            'other_relevant_information': 6
        }
    }
}

MODEL_CONFIG = {
    'efficientnet': {'name': 'efficientnet_b0', 'input_size': 224, 'batch_size': 32, 'lr': 1e-4},
    'vit':          {'name': 'vit_base_patch16_384', 'input_size': 384, 'batch_size': 16, 'lr': 5e-5}
}

TRAIN_CONFIG = {
    'max_epochs': 50,
    'early_stopping_patience': 5,
    'weight_decay': 0.01,
    'num_workers': 2,
    'mixed_precision': True
}

NUM_CLASSES = TASK_CONFIG[TASK]['num_classes']
LABEL_COL   = TASK_CONFIG[TASK]['label_col']
CLASS_NAMES = TASK_CONFIG[TASK]['class_names']
LABEL_MAP   = TASK_CONFIG[TASK]['label_map']

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"Task          : {TASK}")
print(f"Num classes   : {NUM_CLASSES}")
print(f"Classes       : {CLASS_NAMES}")
print(f"Subset size   : {'ALL DATA' if SUBSET_SIZE is None else SUBSET_SIZE}")
print(f"Device        : {device}")
print(f"Base path     : {BASE_PATH}")
print("=" * 60)

In [ ]:
# ========================================
# CELL 5: Load Annotations
# ========================================

def load_annotations(annotations_path, datasplit_path, task, subset_size=None):
    print("\nLoading annotations...")

    all_dfs = []
    for tsv_file in tqdm(list(Path(annotations_path).glob('*.tsv')), desc="Loading TSV"):
        try:
            df = pd.read_csv(tsv_file, sep='\t', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(tsv_file, sep='\t', encoding='latin-1')
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"Total annotations: {len(combined_df)}")

    task_prefix = 'informative' if task == 'informative' else 'humanitarian'
    try:
        train_df = pd.read_csv(
            os.path.join(datasplit_path, f'task_{task_prefix}_text_img_train.tsv'),
            sep='\t', encoding='latin-1')
        val_df = pd.read_csv(
            os.path.join(datasplit_path, f'task_{task_prefix}_text_img_dev.tsv'),
            sep='\t', encoding='latin-1')
        test_df = pd.read_csv(
            os.path.join(datasplit_path, f'task_{task_prefix}_text_img_test.tsv'),
            sep='\t', encoding='latin-1')
    except Exception as e:
        print(f"Error loading datasplit: {e}")
        print(f"Files: {os.listdir(datasplit_path)}")
        raise

    label_col = TASK_CONFIG[task]['label_col']
    if task == 'informative':
        filtered_df = combined_df[
            combined_df[label_col].isin(['informative', 'not_informative'])].copy()
    else:
        filtered_df = combined_df[
            (combined_df['image_info'] == 'informative') &
            (combined_df[label_col].notna())].copy()

    print(f"Filtered annotations: {len(filtered_df)}")

    splits = {}
    for split_name, split_src in [('train', train_df), ('val', val_df), ('test', test_df)]:
        ids = split_src['image_id'].tolist()
        split_df = filtered_df[filtered_df['image_id'].isin(ids)].copy()
        if subset_size and len(split_df) > subset_size:
            split_df = split_df.sample(n=subset_size, random_state=42)
        splits[split_name] = split_df
        print(f"  {split_name}: {len(split_df)} samples")

    print("\nLabel distribution:")
    for name, df in splits.items():
        print(f"\n{name}:")
        print(df[label_col].value_counts())

    return splits

data_splits = load_annotations(ANNOTATIONS_PATH, DATASPLIT_PATH, TASK, SUBSET_SIZE)


In [ ]:
# ========================================
# CELL 6: Dataset Class
# ========================================

class CrisisMMDDataset(Dataset):
    def __init__(self, df, data_image_path, label_col, label_map, transform=None):
        self.df = df.reset_index(drop=True)
        self.data_image_path = data_image_path
        self.label_col = label_col
        self.label_map = label_map
        self.transform = transform
        self.df = self.df[self.df[label_col].notna()].copy()
        self.df = self.df[self.df[label_col].isin(label_map.keys())].copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.data_image_path, row['image_path'])
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        label = self.label_map[row[self.label_col]]
        return image, label

In [ ]:
# ========================================
# CELL 7: Transforms & DataLoaders
# ========================================

def get_transforms(input_size, is_training=True):
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
    if is_training:
        return transforms.Compose([
            transforms.RandomResizedCrop(input_size, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            normalize
        ])
    return transforms.Compose([
        transforms.Resize(int(input_size * 1.14)),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        normalize
    ])


def create_dataloaders(data_splits, model_type, batch_size, num_workers=2):
    input_size = MODEL_CONFIG[model_type]['input_size']
    datasets = {
        split: CrisisMMDDataset(
            data_splits[split], BASE_PATH, LABEL_COL, LABEL_MAP,
            transform=get_transforms(input_size, is_training=(split == 'train'))
        )
        for split in ['train', 'val', 'test']
    }
    loaders = {
        split: DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(split == 'train'),
            num_workers=num_workers,
            pin_memory=True
        )
        for split, ds in datasets.items()
    }
    print(f"\nDataLoaders ({model_type}):")
    for k, v in loaders.items():
        print(f"  {k}: {len(v)} batches")
    return loaders

In [ ]:
# ========================================
# CELL 8: Model Creation
# ========================================

def create_model(model_type, num_classes, pretrained=True):
    model_name = MODEL_CONFIG[model_type]['name']
    print(f"\nCreating {model_type}: {model_name}, classes={num_classes}")
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
    model = model.to(device)
    total  = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params: {total:,} | Trainable: {trainable:,}")
    return model

In [ ]:
# ========================================
# CELL 9: Training Utilities
# ========================================

class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val = self.avg = self.sum = self.count = 0
    def update(self, val, n=1):
        self.val = val; self.sum += val * n; self.count += n
        self.avg = self.sum / self.count


class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience; self.counter = 0
        self.best_loss = None; self.early_stop = False
    def __call__(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss:
            self.best_loss = val_loss; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


def save_checkpoint(model, optimizer, epoch, val_loss, val_acc, filepath):
    torch.save({
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss': val_loss, 'val_acc': val_acc
    }, filepath)
    print(f"  💾 Checkpoint saved: {filepath}")

In [ ]:
# ========================================
# CELL 10: Train One Epoch
# ========================================

def train_one_epoch(model, dataloader, criterion, optimizer, scaler):
    model.train()
    losses, accs = AverageMeter(), AverageMeter()
    pbar = tqdm(dataloader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        with torch.cuda.amp.autocast(enabled=TRAIN_CONFIG['mixed_precision']):
            outputs = model(images)
            loss = criterion(outputs, labels)
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        _, preds = torch.max(outputs, 1)
        losses.update(loss.item(), images.size(0))
        accs.update((preds == labels).float().mean().item(), images.size(0))
        pbar.set_postfix(loss=f'{losses.avg:.4f}', acc=f'{accs.avg:.4f}')
    return losses.avg, accs.avg


@torch.no_grad()
def validate(model, dataloader, criterion):
    model.eval()
    losses, accs = AverageMeter(), AverageMeter()
    all_preds, all_labels = [], []
    pbar = tqdm(dataloader, desc='Validation', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        _, preds = torch.max(outputs, 1)
        losses.update(loss.item(), images.size(0))
        accs.update((preds == labels).float().mean().item(), images.size(0))
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f'{losses.avg:.4f}', acc=f'{accs.avg:.4f}')
    return losses.avg, accs.avg, np.array(all_preds), np.array(all_labels)


In [ ]:
# ========================================
# CELL 11: Main Training Loop
# ========================================

def train_model(model, dataloaders, model_name, num_epochs=50,
                lr=1e-4, weight_decay=0.01, patience=5):
    print(f"\n{'='*60}\nTraining: {model_name}\n{'='*60}")
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.cuda.amp.GradScaler(enabled=TRAIN_CONFIG['mixed_precision'])
    early_stop = EarlyStopping(patience=patience)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_acc, best_epoch = 0.0, 0
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'{model_name}_best.pth')

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}  lr={optimizer.param_groups[0]['lr']:.6f}")
        train_loss, train_acc = train_one_epoch(model, dataloaders['train'], criterion, optimizer, scaler)
        val_loss, val_acc, _, _ = validate(model, dataloaders['val'], criterion)
        scheduler.step()

        for k, v in [('train_loss', train_loss), ('train_acc', train_acc),
                     ('val_loss', val_loss), ('val_acc', val_acc),
                     ('lr', optimizer.param_groups[0]['lr'])]:
            history[k].append(v)

        print(f"  Train  → loss: {train_loss:.4f}  acc: {train_acc:.4f}")
        print(f"  Val    → loss: {val_loss:.4f}  acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc, best_epoch = val_acc, epoch + 1
            save_checkpoint(model, optimizer, epoch, val_loss, val_acc, checkpoint_path)
            print(f"  ✅ New best! val_acc={val_acc:.4f}")

        early_stop(val_loss)
        if early_stop.early_stop:
            print(f"\n⏹ Early stopping at epoch {epoch+1}")
            break

    print(f"\n{'='*60}")
    print(f"Training done! Best val_acc={best_val_acc:.4f} at epoch {best_epoch}")
    print(f"{'='*60}")
    return history

In [ ]:
# ========================================
# CELL 12: Train EfficientNet-B0
# ========================================

efficientnet_model    = create_model('efficientnet', NUM_CLASSES, pretrained=True)
efficientnet_loaders  = create_dataloaders(
    data_splits, 'efficientnet',
    batch_size=MODEL_CONFIG['efficientnet']['batch_size'],
    num_workers=TRAIN_CONFIG['num_workers']
)
efficientnet_history  = train_model(
    efficientnet_model, efficientnet_loaders,
    model_name=f'efficientnet_b0_{TASK}',
    num_epochs=TRAIN_CONFIG['max_epochs'],
    lr=MODEL_CONFIG['efficientnet']['lr'],
    weight_decay=TRAIN_CONFIG['weight_decay'],
    patience=TRAIN_CONFIG['early_stopping_patience']
)

In [ ]:
# ========================================
# CELL 13: Train ViT-B/16
# ========================================

vit_model   = create_model('vit', NUM_CLASSES, pretrained=True)
vit_loaders = create_dataloaders(
    data_splits, 'vit',
    batch_size=MODEL_CONFIG['vit']['batch_size'],
    num_workers=TRAIN_CONFIG['num_workers']
)
vit_history = train_model(
    vit_model, vit_loaders,
    model_name=f'vit_b16_{TASK}',
    num_epochs=TRAIN_CONFIG['max_epochs'],
    lr=MODEL_CONFIG['vit']['lr'],
    weight_decay=TRAIN_CONFIG['weight_decay'],
    patience=TRAIN_CONFIG['early_stopping_patience']
)

In [ ]:
# ========================================
# CELL 14: Plot Training History
# ========================================

def plot_training_history(histories, model_names, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    for history, name in zip(histories, model_names):
        axes[0, 0].plot(history['train_loss'], label=f'{name} Train')
        axes[0, 0].plot(history['val_loss'], '--', label=f'{name} Val')
        axes[0, 1].plot(history['train_acc'], label=f'{name} Train')
        axes[0, 1].plot(history['val_acc'], '--', label=f'{name} Val')
        axes[1, 0].plot(history['lr'], label=name)
    for ax, title, ylabel in zip(
        [axes[0,0], axes[0,1], axes[1,0]],
        ['Loss', 'Accuracy', 'Learning Rate'],
        ['Loss', 'Accuracy', 'LR']
    ):
        ax.set_title(title); ax.set_ylabel(ylabel)
        ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
    axes[1, 0].set_yscale('log')

    best_accs = [max(h['val_acc']) for h in histories]
    axes[1, 1].bar(model_names, best_accs, color=['#1f77b4', '#ff7f0e'])
    axes[1, 1].set_title('Best Val Accuracy'); axes[1, 1].grid(True, axis='y')
    for i, v in enumerate(best_accs):
        axes[1, 1].text(i, v + 0.005, f'{v:.4f}', ha='center')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

plot_training_history(
    [efficientnet_history, vit_history],
    ['EfficientNet-B0', 'ViT-B/16'],
    os.path.join(RESULTS_DIR, 'training_history.png')
)

In [ ]:
# ========================================
# CELL 15: Evaluation Function
# ========================================

@torch.no_grad()
def evaluate_model(model, dataloader, class_names):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(dataloader, desc='Evaluating'):
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

    all_preds, all_labels, all_probs = map(np.array, [all_preds, all_labels, all_probs])
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0)
    cm     = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds,
                                   target_names=class_names, zero_division=0)
    return {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'confusion_matrix': cm, 'classification_report': report,
        'predictions': all_preds, 'labels': all_labels, 'probabilities': all_probs
    }


def plot_confusion_matrix(cm, class_names, title, save_path):
    plt.figure(figsize=(max(8, len(class_names)*1.5), max(6, len(class_names)*1.2)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

In [ ]:
# ========================================
# CELL 16: Evaluate EfficientNet-B0
# ========================================

ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f'efficientnet_b0_{TASK}_best.pth'))
efficientnet_model.load_state_dict(ckpt['model_state_dict'])

efficientnet_results = evaluate_model(efficientnet_model, efficientnet_loaders['test'], CLASS_NAMES)
print("\nEfficientNet-B0 Test Results")
print(f"  Accuracy : {efficientnet_results['accuracy']:.4f}")
print(f"  Precision: {efficientnet_results['precision']:.4f}")
print(f"  Recall   : {efficientnet_results['recall']:.4f}")
print(f"  F1-Score : {efficientnet_results['f1']:.4f}")
print(efficientnet_results['classification_report'])

plot_confusion_matrix(
    efficientnet_results['confusion_matrix'], CLASS_NAMES,
    'EfficientNet-B0 Confusion Matrix',
    os.path.join(RESULTS_DIR, 'efficientnet_confusion_matrix.png')
)

In [ ]:
# ========================================
# CELL 17: Evaluate ViT-B/16
# ========================================

ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f'vit_b16_{TASK}_best.pth'))
vit_model.load_state_dict(ckpt['model_state_dict'])

vit_results = evaluate_model(vit_model, vit_loaders['test'], CLASS_NAMES)
print("\nViT-B/16 Test Results")
print(f"  Accuracy : {vit_results['accuracy']:.4f}")
print(f"  Precision: {vit_results['precision']:.4f}")
print(f"  Recall   : {vit_results['recall']:.4f}")
print(f"  F1-Score : {vit_results['f1']:.4f}")
print(vit_results['classification_report'])

plot_confusion_matrix(
    vit_results['confusion_matrix'], CLASS_NAMES,
    'ViT-B/16 Confusion Matrix',
    os.path.join(RESULTS_DIR, 'vit_confusion_matrix.png')
)


In [ ]:
# ========================================
# CELL 18: Ensemble — Simple & Weighted
# ========================================

eff_probs   = efficientnet_results['probabilities']
vit_probs   = vit_results['probabilities']
test_labels = efficientnet_results['labels']

# Weights berdasarkan val accuracy
eff_val_acc = max(efficientnet_history['val_acc'])
vit_val_acc = max(vit_history['val_acc'])
total_acc   = eff_val_acc + vit_val_acc
w_eff, w_vit = eff_val_acc / total_acc, vit_val_acc / total_acc
print(f"\nEnsemble weights: EfficientNet={w_eff:.4f}, ViT={w_vit:.4f}")

def ensemble_metrics(probs, labels, class_names, method_name, save_path):
    preds = np.argmax(probs, axis=1)
    acc  = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    cm   = confusion_matrix(labels, preds)
    print(f"\n{method_name}: Acc={acc:.4f}  P={prec:.4f}  R={rec:.4f}  F1={f1:.4f}")
    plot_confusion_matrix(cm, class_names, f'{method_name} Confusion Matrix', save_path)
    return acc, prec, rec, f1, cm

simple_probs   = (eff_probs + vit_probs) / 2
weighted_probs = w_eff * eff_probs + w_vit * vit_probs

(simple_acc, simple_prec, simple_rec, simple_f1, _) = ensemble_metrics(
    simple_probs, test_labels, CLASS_NAMES,
    'Ensemble (Simple)', os.path.join(RESULTS_DIR, 'ensemble_simple_cm.png'))
(weighted_acc, weighted_prec, weighted_rec, weighted_f1, _) = ensemble_metrics(
    weighted_probs, test_labels, CLASS_NAMES,
    'Ensemble (Weighted)', os.path.join(RESULTS_DIR, 'ensemble_weighted_cm.png'))


In [ ]:
# ========================================
# CELL 19: Efficiency Metrics
# ========================================

def measure_inference_time(model, input_size, num_runs=100, warmup=10):
    model.eval()
    dummy = torch.randn(1, 3, input_size, input_size).to(device)
    with torch.no_grad():
        for _ in range(warmup): model(dummy)
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            model(dummy); torch.cuda.synchronize()
    return (time.time() - t0) / num_runs * 1000


def measure_memory_usage(model, input_size):
    model.eval()
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    dummy = torch.randn(1, 3, input_size, input_size).to(device)
    with torch.no_grad(): model(dummy)
    return torch.cuda.max_memory_allocated() / (1024 ** 2)


def count_flops(model, input_size):
    from thop import profile
    dummy = torch.randn(1, 3, input_size, input_size).to(device)
    flops, params = profile(model, inputs=(dummy,), verbose=False)
    return flops / 1e9, params / 1e6


print("\nMeasuring efficiency metrics...")
eff_time   = measure_inference_time(efficientnet_model, 224)
vit_time   = measure_inference_time(vit_model, 384)
ens_time   = eff_time + vit_time
eff_mem    = measure_memory_usage(efficientnet_model, 224)
vit_mem    = measure_memory_usage(vit_model, 384)
ens_mem    = eff_mem + vit_mem
eff_flops, eff_params = count_flops(efficientnet_model, 224)
vit_flops, vit_params = count_flops(vit_model, 384)
ens_flops  = eff_flops + vit_flops
ens_params = eff_params + vit_params

print(f"EfficientNet-B0 : {eff_time:.2f}ms  {eff_mem:.0f}MB  {eff_flops:.2f}G  {eff_params:.1f}M params")
print(f"ViT-B/16        : {vit_time:.2f}ms  {vit_mem:.0f}MB  {vit_flops:.2f}G  {vit_params:.1f}M params")
print(f"Ensemble        : {ens_time:.2f}ms  {ens_mem:.0f}MB  {ens_flops:.2f}G  {ens_params:.1f}M params")


In [ ]:
# ========================================
# CELL 20: Summary Table & Trade-off Plot
# ========================================

results_df = pd.DataFrame({
    'Model': ['EfficientNet-B0', 'ViT-B/16', 'Ensemble (Simple)', 'Ensemble (Weighted)'],
    'Accuracy':  [efficientnet_results['accuracy'], vit_results['accuracy'], simple_acc, weighted_acc],
    'Precision': [efficientnet_results['precision'], vit_results['precision'], simple_prec, weighted_prec],
    'Recall':    [efficientnet_results['recall'], vit_results['recall'], simple_rec, weighted_rec],
    'F1-Score':  [efficientnet_results['f1'], vit_results['f1'], simple_f1, weighted_f1],
    'Time (ms)': [eff_time, vit_time, ens_time, ens_time],
    'Memory (MB)': [eff_mem, vit_mem, ens_mem, ens_mem],
    'FLOPs (G)': [eff_flops, vit_flops, ens_flops, ens_flops],
    'Params (M)': [eff_params, vit_params, ens_params, ens_params],
})

print("\n" + "="*80 + "\nSUMMARY RESULTS\n" + "="*80)
print(results_df.to_string(index=False))

results_df.to_csv(os.path.join(RESULTS_DIR, 'summary_results.csv'), index=False)

# Trade-off plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
markers = ['o', 's', '^', 'D']
for ax, xcol, xlabel in zip(
    axes,
    ['Time (ms)', 'Memory (MB)', 'FLOPs (G)'],
    ['Inference Time (ms)', 'Memory (MB)', 'FLOPs (GFLOPs)']
):
    for i, row in results_df.iterrows():
        ax.scatter(row[xcol], row['Accuracy'], s=200,
                   c=colors[i], marker=markers[i], label=row['Model'], alpha=0.8, zorder=5)
        ax.annotate(row['Model'], (row[xcol], row['Accuracy']),
                    textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
    ax.set_xlabel(xlabel); ax.set_ylabel('Accuracy')
    ax.set_title(f'Accuracy vs {xlabel}')
    ax.grid(True, alpha=0.3); ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tradeoff_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ========================================
# CELL 21: Save All Results JSON
# ========================================

all_results = {
    'task': TASK, 'subset_size': SUBSET_SIZE,
    'num_classes': NUM_CLASSES, 'class_names': CLASS_NAMES,
    'efficientnet': {
        **{k: float(v) for k, v in {
            'accuracy': efficientnet_results['accuracy'],
            'precision': efficientnet_results['precision'],
            'recall': efficientnet_results['recall'],
            'f1': efficientnet_results['f1'],
            'inference_time_ms': eff_time,
            'memory_mb': eff_mem, 'flops_g': eff_flops, 'params_m': eff_params
        }.items()},
        'history': {k: [float(x) for x in v]
                    for k, v in efficientnet_history.items() if k != 'lr'}
    },
    'vit': {
        **{k: float(v) for k, v in {
            'accuracy': vit_results['accuracy'],
            'precision': vit_results['precision'],
            'recall': vit_results['recall'],
            'f1': vit_results['f1'],
            'inference_time_ms': vit_time,
            'memory_mb': vit_mem, 'flops_g': vit_flops, 'params_m': vit_params
        }.items()},
        'history': {k: [float(x) for x in v]
                    for k, v in vit_history.items() if k != 'lr'}
    },
    'ensemble_simple': {k: float(v) for k, v in {
        'accuracy': simple_acc, 'precision': simple_prec,
        'recall': simple_rec, 'f1': simple_f1,
        'inference_time_ms': ens_time, 'memory_mb': ens_mem,
        'flops_g': ens_flops, 'params_m': ens_params
    }.items()},
    'ensemble_weighted': {
        **{k: float(v) for k, v in {
            'accuracy': weighted_acc, 'precision': weighted_prec,
            'recall': weighted_rec, 'f1': weighted_f1,
            'inference_time_ms': ens_time, 'memory_mb': ens_mem,
            'flops_g': ens_flops, 'params_m': ens_params
        }.items()},
        'weights': [float(w_eff), float(w_vit)]
    }
}

with open(os.path.join(RESULTS_DIR, 'all_results.json'), 'w') as f:
    json.dump(all_results, f, indent=2)

print("\n" + "="*60)
print("✅ EXPERIMENT SELESAI!")
print("="*60)
print(f"\nOutput tersimpan di: {OUTPUT_DIR}")
print(f"  checkpoints/   → model terbaik (.pth)")
print(f"  results/       → CSV, JSON, PNG plots")

best_acc = results_df['Accuracy'].max()
best_mdl = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
print(f"\n🏆 Best Model: {best_mdl} — Accuracy: {best_acc:.4f}")


